In [1]:
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.metrics import(
roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix,)

In [2]:
data_path = 'targets_features_split.parquet'

num_trials_cancel = 100
num_trials_delay = 100

n_estimators_max = 3000
early_stopping_rounds=100

# threshold > 0.5
threshold_grid = np.linspace(0.1, 0.9, 81)



In [3]:
cols= pq.ParquetFile(data_path).schema.names

id_cols = ['FlightDate', 'split']
target_cols = ['target', 'target_cancelled', 'target_delayed', 'target_delayed_non_cancelled']
feature_cols = [c for c in cols if c not in (id_cols + target_cols)]


In [4]:
# load in cols
all_cols = ['split', 'target_cancelled', 'target_delayed_non_cancelled'] + feature_cols

df = pd.read_parquet(data_path, columns=all_cols)
train = df['split'] == 'train'
val = df['split'] == 'val'
test = df['split'] == 'test'

In [5]:
def compute_class_weight_ratio(y):
    y = np.asarray(y).astype(int)
    pos = int(y.sum())
    neg = int((1-y).sum())
    return neg/ max(pos, 1)

# parameter tuning

def param_combo(rng):
    return {
        'max_depth': int(rng.integers(4, 11)),
        'min_child_weight': float(rng.choice([1, 2, 3, 5, 8, 10])),
        'learning_rate': float(rng.choice([0.01, 0.02, 0.03, 0.05, 0.08, 0.1])),
        'subsample': float(rng.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
        'colsample_bytree': float(rng.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
        'gamma': float(rng.choice([0.0, 0.1, 0.3, 0.5, 1.0, 2.0])),
        'reg_alpha': float(rng.choice([0.0, 1e-3, 1e-2, 1e-1, 1.0])),
        'reg_lambda': float(rng.choice([0.5, 1.0, 2.0, 5.0, 10.0])),
    }

def best_threshold_for_f1(y_true, y_prob, grid):
    best_thr = 0.5
    best_f1 = -1.0
    for thr in grid:
        y_pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, y_pred)
        if score > best_f1:
            best_f1 = score
            best_thr = float(thr)
    return best_thr, best_f1


def print_binary_metrics(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    print(f'\n{name}')
    print(f'ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}')
    print(f'PR-AUC: {average_precision_score(y_true, y_prob):.4f}')
    print(f'F1: {f1_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'Recall: {recall_score(y_true, y_pred):.4f}')
    print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
    print('Confusion matrix:')
    print(confusion_matrix(y_true, y_pred))


def run_random_search(task_name, X_train, y_train, X_val, y_val, n_trials, random_state=42):
    rng = np.random.default_rng(random_state)
    class_weight = compute_class_weight_ratio(y_train)

    best = {
        'trial': None,
        'val_pr_auc': -1.0,
        'val_roc_auc': -1.0,
        'val_best_f1': -1.0,
        'val_best_threshold': 0.5,
        'params': None,
        'best_iteration': None,
        'model': None,
    }

    rows = []
    t0 = time.time()

    for i in range(1, n_trials + 1):
        params = param_combo(rng)

        model = XGBClassifier(
            n_estimators=n_estimators_max,
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            n_jobs=-1,
            random_state=random_state,
            scale_pos_weight=class_weight,
            early_stopping_rounds=early_stopping_rounds,
            **params,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        p_val = model.predict_proba(X_val)[:, 1]
        val_pr_auc = float(average_precision_score(y_val, p_val))
        val_roc_auc = float(roc_auc_score(y_val, p_val))
        thr, val_best_f1 = best_threshold_for_f1(y_val, p_val, threshold_grid)

        row = {
            'task': task_name,
            'trial': i,
            'val_pr_auc': val_pr_auc,
            'val_roc_auc': val_roc_auc,
            'val_best_f1': float(val_best_f1),
            'val_best_threshold': float(thr),
            'best_iteration': int(model.best_iteration) if hasattr(model, 'best_iteration') and model.best_iteration is not None else None,
            **params,
        }
        rows.append(row)

        better = (val_pr_auc > best['val_pr_auc']) or (
            np.isclose(val_pr_auc, best['val_pr_auc']) and val_best_f1 > best['val_best_f1']
        )

        if better:
            best['trial'] = i
            best['val_pr_auc'] = val_pr_auc
            best['val_roc_auc'] = val_roc_auc
            best['val_best_f1'] = float(val_best_f1)
            best['val_best_threshold'] = float(thr)
            best['params'] = params
            best['best_iteration'] = int(model.best_iteration) if hasattr(model, 'best_iteration') and model.best_iteration is not None else None
            best['model'] = model

        if i % 10 == 0 or i == 1:
            elapsed_min = (time.time() - t0) / 60
            print(
                f"[{task_name}] trial={i}/{n_trials} "
                f"val_pr_auc={val_pr_auc:.4f} val_roc_auc={val_roc_auc:.4f} "
                f"best_pr_auc={best['val_pr_auc']:.4f} elapsed_min={elapsed_min:.1f}"
            )

    history = pd.DataFrame(rows).sort_values('val_pr_auc', ascending=False).reset_index(drop=True)
    return best, history

In [6]:
# Model A
X_train_A = df.loc[train, feature_cols]
X_val_A = df.loc[val, feature_cols]
X_test_A = df.loc[test, feature_cols]
y_train_A = df.loc[train, 'target_cancelled'].astype(int)
y_val_A = df.loc[val, 'target_cancelled'].astype(int)
y_test_A = df.loc[test, 'target_cancelled'].astype(int)

best_A, history_A = run_random_search(
    task_name='cancel',
    X_train=X_train_A,
    y_train=y_train_A,
    X_val=X_val_A,
    y_val=y_val_A,
    n_trials=num_trials_cancel,
    random_state=42,
)
history_A.to_csv('tuning_history_model_A_cancel.csv', index=False)

# Model B 
train_B = train & (df['target_delayed_non_cancelled'] >= 0)
val_B = val & (df['target_delayed_non_cancelled'] >= 0)
test_B = test & (df['target_delayed_non_cancelled'] >= 0)

X_train_B = df.loc[train_B, feature_cols]
X_val_B = df.loc[val_B, feature_cols]
X_test_B = df.loc[test_B, feature_cols]
y_train_B = df.loc[train_B, 'target_delayed_non_cancelled'].astype(int)
y_val_B = df.loc[val_B, 'target_delayed_non_cancelled'].astype(int)
y_test_B = df.loc[test_B, 'target_delayed_non_cancelled'].astype(int)

best_B, history_B = run_random_search(
    task_name='delay_non_cancelled',
    X_train=X_train_B,
    y_train=y_train_B,
    X_val=X_val_B,
    y_val=y_val_B,
    n_trials=num_trials_delay,
    random_state=49,
)
history_B.to_csv('tuning_history_model_B_delay.csv', index=False)

print('Both tuning runs finished.')


[cancel] trial=1/100 val_pr_auc=0.1312 val_roc_auc=0.7619 best_pr_auc=0.1312 elapsed_min=2.2


[cancel] trial=10/100 val_pr_auc=0.1237 val_roc_auc=0.7637 best_pr_auc=0.1318 elapsed_min=25.7


[cancel] trial=20/100 val_pr_auc=0.1097 val_roc_auc=0.7584 best_pr_auc=0.1318 elapsed_min=54.7


[cancel] trial=30/100 val_pr_auc=0.1304 val_roc_auc=0.7599 best_pr_auc=0.1332 elapsed_min=81.1


[cancel] trial=40/100 val_pr_auc=0.1313 val_roc_auc=0.7604 best_pr_auc=0.1332 elapsed_min=107.9


...

[delay_non_cancelled] trial=60/100 val_pr_auc=0.3618 val_roc_auc=0.7091 best_pr_auc=0.3640 elapsed_min=286.8


[delay_non_cancelled] trial=70/100 val_pr_auc=0.3610 val_roc_auc=0.7071 best_pr_auc=0.3640 elapsed_min=340.2


[delay_non_cancelled] trial=80/100 val_pr_auc=0.3634 val_roc_auc=0.7084 best_pr_auc=0.3640 elapsed_min=385.1


[delay_non_cancelled] trial=90/100 val_pr_auc=0.3619 val_roc_auc=0.7066 best_pr_auc=0.3640 elapsed_min=427.6


[delay_non_cancelled] trial=100/100 val_pr_auc=0.3595 val_roc_auc=0.7053 best_pr_auc=0.3640 elapsed_min=462.7
Both tuning runs finished.
